# Chapter 16 — Natural Language Processing with RNNs and Attention
Companion notebook to `part2_deep_learning/16_nlp_attention_transformers.md`.
Downloads the IMDB dataset the first time (via Keras).

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras

## Sentiment analysis: Embedding + GRU on IMDB reviews

In [ ]:
vocab_size = 10000
(X_train, y_train), (X_test, y_test) = keras.datasets.imdb.load_data(num_words=vocab_size)

X_train = keras.preprocessing.sequence.pad_sequences(X_train, maxlen=200)
X_test = keras.preprocessing.sequence.pad_sequences(X_test, maxlen=200)

model = keras.models.Sequential([
    keras.layers.Embedding(vocab_size, 32, input_shape=[None], mask_zero=True),
    keras.layers.GRU(32, return_sequences=True),
    keras.layers.GRU(32),
    keras.layers.Dense(1, activation="sigmoid"),
])
model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])
history = model.fit(X_train, y_train, epochs=3, validation_split=0.2, batch_size=64)
print("Test accuracy:", model.evaluate(X_test, y_test)[1])

## A minimal self-attention block (the core of the Transformer)

In [ ]:
# keras.layers.MultiHeadAttention implements Scaled Dot-Product Attention with multiple heads.
seq_len, d_model, num_heads = 10, 32, 4

sample_input = tf.random.normal([2, seq_len, d_model])   # (batch, seq_len, d_model)

mha = keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model // num_heads)
# self-attention: query, value, and key are all the same sequence
attn_output = mha(query=sample_input, value=sample_input, key=sample_input)
print("Self-attention output shape:", attn_output.shape)   # same shape as input

## Positional encoding (sine/cosine), so attention knows word order

In [ ]:
def positional_encoding(max_len, d_model):
    positions = np.arange(max_len)[:, np.newaxis]
    dims = np.arange(d_model)[np.newaxis, :]
    angle_rates = 1 / np.power(10000, (2 * (dims // 2)) / np.float32(d_model))
    angle_rads = positions * angle_rates
    angle_rads[:, 0::2] = np.sin(angle_rads[:, 0::2])
    angle_rads[:, 1::2] = np.cos(angle_rads[:, 1::2])
    return tf.cast(angle_rads, dtype=tf.float32)

pe = positional_encoding(50, 32)
import matplotlib.pyplot as plt
plt.pcolormesh(pe.numpy(), cmap="RdBu")
plt.xlabel("Depth"); plt.ylabel("Position"); plt.colorbar()
plt.title("Positional encoding")
plt.show()

## ✍️ Practice
1. Explain why a plain Encoder-Decoder (no attention) struggles more with long sentences than short ones.
2. Above: try a Bidirectional wrapper (`keras.layers.Bidirectional(keras.layers.GRU(32))`) — does accuracy improve?
3. Debug: a Transformer decoder without a causal mask trains with suspiciously great loss but generates poorly — why?
4. Why does the Transformer need positional encoding when RNNs never needed anything like it?

In [ ]:
# Your code / notes here